# Data prep — HK physical tiles (YOLO26n)

Merge the Roboflow YOLO26 zips in `data/raw/` into a **43-class** Hong Kong schema:

```
data/processed/hk_merged/
  train|val|test / images + labels
  data.yaml
```

Select kernel **Python (mjss)**.

Class map + remaps: `configs/hk_tile_map.yaml`.

In [ ]:
from pathlib import Path

import torch
import yaml

ROOT = Path.cwd()
if not (ROOT / "configs").exists() and (ROOT.parent / "configs").exists():
    ROOT = ROOT.parent

MERGED = ROOT / "data" / "processed" / "hk_merged"
MAP = ROOT / "configs" / "hk_tile_map.yaml"

print("torch", torch.__version__, "| MPS available:", torch.backends.mps.is_available())
print("ROOT", ROOT)
print("map exists:", MAP.exists(), "| merged exists:", MERGED.exists())

## 1. Build merged dataset

Run once (or after adding new zips). Takes a few minutes.

In [ ]:
# Uncomment to (re)build:
# !python scripts/merge_hk_dataset.py --clean
print("Run from project root: python scripts/merge_hk_dataset.py --clean")

## 2. Inspect counts + class histogram

In [ ]:
from collections import Counter

cfg = yaml.safe_load((MERGED / "data.yaml").read_text()) if (MERGED / "data.yaml").exists() else {}
names = cfg.get("names") or yaml.safe_load(MAP.read_text())["names"]
print("nc:", len(names))
print("names:", names)


def count_split(split: str) -> dict:
    img_dir = MERGED / split / "images"
    lbl_dir = MERGED / split / "labels"
    imgs = list(img_dir.glob("*")) if img_dir.exists() else []
    lbls = list(lbl_dir.glob("*.txt")) if lbl_dir.exists() else []
    boxes = Counter()
    for f in lbls:
        for line in f.read_text().splitlines():
            if line.strip():
                boxes[names[int(line.split()[0])]] += 1
    return {"images": len(imgs), "labels": len(lbls), "boxes": sum(boxes.values()), "by_class": boxes}


for split in ("train", "val", "test"):
    stats = count_split(split)
    print(f"\n{split}: images={stats['images']} labels={stats['labels']} boxes={stats['boxes']}")
    rare = [(n, stats["by_class"][n]) for n in names if stats["by_class"][n] < 50]
    if rare:
        print("  rare (<50):", rare)

## 3. Spot-check label overlays

In [ ]:
import random

import matplotlib.pyplot as plt
from PIL import Image, ImageDraw

split = "val"
lbl_dir = MERGED / split / "labels"
img_dir = MERGED / split / "images"
samples = random.sample(list(lbl_dir.glob("*.txt")), k=min(4, len(list(lbl_dir.glob("*.txt")))))

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
for ax, lbl in zip(axes.flat, samples):
    imgs = list(img_dir.glob(lbl.stem + ".*"))
    im = Image.open(imgs[0]).convert("RGB")
    draw = ImageDraw.Draw(im)
    W, H = im.size
    for line in lbl.read_text().splitlines():
        parts = line.split()
        if len(parts) < 5:
            continue
        cid, cx, cy, w, h = int(parts[0]), *map(float, parts[1:5])
        x1, y1 = (cx - w / 2) * W, (cy - h / 2) * H
        x2, y2 = (cx + w / 2) * W, (cy + h / 2) * H
        draw.rectangle([x1, y1, x2, y2], outline=(255, 0, 0), width=2)
        draw.text((x1, max(0, y1 - 12)), names[cid], fill=(255, 0, 0))
    ax.imshow(im)
    ax.set_title(lbl.stem[:40])
    ax.axis("off")
plt.tight_layout()
plt.show()

## 4. Train

```bash
source .venv/bin/activate
python scripts/train.py
```

In [ ]:
print("Ready. Train with: python scripts/train.py")
print("data yaml:", MERGED / "data.yaml")